# Relaxation of Structure

Run this cell to install DFTpy in google colab

```bash
!python -m pip install "git+https://github.com/Quantum-MultiScale/DFTpy.git@dev"
```

Download the pseudopotential file

```bash
!wget https://raw.githubusercontent.com/Quantum-MultiScale/DFTpy/dev/DATA/Al_lda.oe01.recpot
```


Perform the structure relaxation is also easy with `ASE`_.
This is one example to relax the force of given structure:

In [2]:
import os
import pathlib
import numpy as np
# from ase.calculators.interface import Calculator
from ase.lattice.cubic import FaceCenteredCubic
from ase.optimize import BFGS, LBFGS, FIRE
from ase.optimize.sciopt import SciPyFminBFGS, SciPyFminCG
from ase.constraints import StrainFilter, UnitCellFilter
from ase.io.trajectory import Trajectory
from ase import units
import ase.io

from dftpy.config import DefaultOption, OptionFormat
from dftpy.interface import OptimizeDensityConf
from dftpy.api.api4ase import DFTpyCalculator




In [3]:
############################## initial config ##############################
conf = DefaultOption()
conf['PATH']['pppath'] = '../DATA/'
conf['PP']['Al'] = 'Al_lda.oe01.recpot'
conf['JOB']['calctype'] = 'Energy Force Stress'
conf['KEDF']['kedf'] = 'WT'
conf = OptionFormat(conf)
#-----------------------------------------------------------------------
path = conf['PATH']['pppath']
atoms = ase.io.read(path+'/'+'fcc.vasp')

calc = DFTpyCalculator(config = conf)
atoms.set_calculator(calc)

trajfile = 'opt.traj'

/var/folders/9b/gsh5m3sd3j96rq9k2zstrc_40000gn/T/ipykernel_16442/3153497339.py:13: FutureWarning: Please use atoms.calc = calc
  atoms.set_calculator(calc)


In [ ]:
af = atoms
af = StrainFilter(atoms)
af = UnitCellFilter(atoms)

opt = SciPyFminCG(af, trajectory = trajfile)

opt.run(fmax = 0.01)

traj = Trajectory(trajfile)


Write the optimize structure in a vasp file

In [7]:
ase.io.write('opt.vasp', traj[-1], direct = True, vasp5 = True)
